The pre-processing to be done for the features that we need for the knn to infer the similar recipes from the database,
They are 1.Ingredients
         2.Tags
         3. Nutrients

In [ ]:
# importing all the core libraries
import pandas as pd
import numpy as np 
import re
import ast

In [ ]:
# pip install num2words

In [ ]:
from num2words import num2words

Create dataframe from the combined datasets

In [ ]:
# the dataset_combined .. is the combined dataset .. from multiple different sources

In [ ]:
df=pd.read_csv("Dataset_combined.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['nutrition'][0]

In [ ]:
#  the nutrition column is in form of list to convert that to string using literal_eval
df['nutrition'] = df['nutrition'].apply(ast.literal_eval)

In [ ]:
# the nutrition column is splitted and reduced into a data frame of columns same as number of elements in the list
nutrition_df=df['nutrition'].apply(pd.Series)

In [ ]:
nutrition_df.head()

In [ ]:
nutrition_df.info()

In [ ]:
print(nutrition_df.columns)


In [ ]:
nutritions={}

In [ ]:
nutritions=nutrition_df.to_dict(orient='list')
nutritions


In [ ]:
nutritions.keys()

In [ ]:
# Above same metric is in different names , there is a need to standardize it

In [ ]:
nutritions.items()

In [ ]:
# a stndard structure for the final dict for nutrients

In [ ]:
nutrients_cleaned={"calories":[], "protein":[], "carbohydrates":[], "fiber":[], "fat":[], "sodium":[]}

Cleaning and exploding the nutrition column to uniques columns for every nutrients

In [ ]:
# Define mapping to unify columns
key_map = {
    "calories": ['Energy', 'kcal', 'Calories ', 'Kilojoules ', ' Calories '],
    "protein": ['Protein', 'protein', 'Protein '],
    "carbohydrates": ['Carbohydrates', 'carbs', 'Carbohydrates ', ' Carbohydrates '],
    "fiber": ['Fiber', 'fibre', 'Dietary fibre ', ' Dietary fibre '],
    "fat": ['Fat', 'fat', 'Total fat ', 'Total Fat'],
    "sodium": ['Sodium', 'sodium', 'Sodium ', ' Sodium ']
}


# Helper to extract number from string ..like for 300 kcl --> 300
def extract_number(val):
    if isinstance(val, str):
        match = re.search(r"[\d.]+", val)
        if match:
            return float(match.group())
    return np.nan

# Get number of rows
num_rows = max(len(v) for v in nutritions.values())

# Initialize cleaned dict with NaNs..we can overwrite this later , for valid entries..
#  helps handle missing values
nutrients_cleaned = {
    key: [np.nan] * num_rows
    for key in key_map
}

# Process and fill cleaned data

'''For every clean nutrient (like "calories"), go through each row.
 Try each possible messy key for that nutrient (like "Energy", "Calories "),
   and as soon as you find a usable value,
 extract the number and assign it. Stop looking after the first valid one.'''

for target, source_keys in key_map.items():
    for i in range(num_rows):
        for src in source_keys:
            if src in nutritions and i < len(nutritions[src]):
                value = nutritions[src][i]
                num = extract_number(value)
                if not np.isnan(num):
                    nutrients_cleaned[target][i] = num
                    break  # Take the first valid value, skip rest

# Create cleaned dataframe
clean_df = pd.DataFrame(nutrients_cleaned)
print(clean_df)


The original nutrition column is dropped from the original table and the new dataframe of nutritions will be concatenated with it .

In [ ]:
df=pd.concat([df.drop('nutrition',axis=1),clean_df],axis=1)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [ ]:
df.info()

In [ ]:
import nltk
from fractions import Fraction
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords #Predefined list of common English words (like "the", "is", "and") 
# — mostly meaningless in modeling.
nltk.download('stopwords')

Pre Processing for the Ingredients and Tags columns

This function returns the tokenized pre processed list of string which is further needed for the vectorization.

In [ ]:
def preprocessing(text):
    def convert_number(match):
        num_str = match.group()
        try:
            # Handle fractions like 1/2
            if '/' in num_str:
                number = float(Fraction(num_str))
            else:
                number = int(num_str)
            return num2words(number)
        except:
            return num_str  # In case of error, return original
    # Join and lowercase
    # text =text.astype()

    text = text.lower()

    # Replace numbers and fractions with words
    text = re.sub(r'\d+(?:/\d+)?', convert_number, text)

    # Clean up punctuations leaves only the lowecase letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    # removes the cooking units
    text = re.sub(r'\b(cups?|tbsp|tsp|pinch|optional|taste)\b', '', text)

    tokens=text.split()
    

    stop_words = set(stopwords.words('english'))
    filtered_tokens = [w for w in tokens if w not in stop_words] # removal of stop words
    return filtered_tokens


In [ ]:
df['ingredients'] = df['ingredients'].astype(str)

In [ ]:
df['ingredients']=df['ingredients'].apply(preprocessing)

In [ ]:
df['tags']=df['tags'].apply(preprocessing)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.to_csv("PreProcessedData.csv")

In [ ]:
df_new=pd.read_csv("PreProcessedData.csv")
df_new.head()